#### Code Converter

In [1]:
# imports

import os
import io
import sys
from dotenv import load_dotenv
from openai import OpenAI
import subprocess
import gradio as gr
from IPython.display import display, Markdown

In [2]:
# validating API keys

load_dotenv(override = True)

gemini_base_url = os.getenv('GEMINI_BASE_URL')
gemini_api_key  = os.getenv('GEMINI_API_KEY')

groq_base_url = os.getenv('GROQ_BASE_URL')
groq_api_key = os.getenv('GROQ_API_KEY')

if gemini_api_key:
    print(f'Gemini API Key found and starts with {gemini_api_key[0:3]}')
else:
    print('Gemini API Key not found')

if groq_api_key:
    print(f'Groq API Key found and starts with {groq_api_key[0:3]}')
else:
    print('Groq API Key not found')

Gemini API Key found and starts with AQ.
Groq API Key found and starts with gsk


In [3]:
# clients

gemini = OpenAI(base_url = gemini_base_url, api_key = gemini_api_key)
groq = OpenAI(base_url = groq_base_url, api_key = groq_api_key)

In [4]:
# models and clients

models = ['gemini-3.6-flash', 'openai/gpt-oss-20b', 'openai/gpt-oss-120b', 'groq/compound-mini']
clients = {"gemini-3.6-flash":gemini, "openai/gpt-oss-20b":groq, "openai/gpt-oss-120b":groq, "groq/compound-mini":groq}

In [5]:
# to retrive system info and tool chain info

from system_info import retrieve_system_info

system_info = retrieve_system_info()

In [6]:
# to request appropriate compile command and run command via LLM
message = f"""
Here is a report of the system information for my computer.
I want to run a C++ compiler to compile a single C++ file called main.cpp and then execute it in the simplest way possible.
Please reply with whether I need to install any C++ compiler to do this. If so, please provide the simplest step by step instructions to do so.

If I'm already set up to compile C++ code, then I'd like to run something like this in Python to compile and execute the code:
```python
compile_command = # something here - to achieve the fastest possible runtime performance
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
run_command = # something here
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
return run_result.stdout
```
Please tell me exactly what I should use for the compile_command and run_command.

System information:
{system_info}
"""

response = gemini.chat.completions.create(model=models[0], messages=[{"role": "user", "content": message}])
display(Markdown(response.choices[0].message.content))

Based on your system information, **no, you do not need to install any C++ compiler.** 

Your system already has **Apple Clang** installed and configured via the macOS Command Line Tools (`xcode-select (CLT)`), accessible via `g++` or `clang++`.

---

### Python Code Setup

Since you are running an Apple M1 chip (`arm64`), you can pass optimization flags to target your CPU architecture directly for the fastest possible execution speed.

Here is exactly what to use for `compile_command` and `run_command`:

```python
import subprocess

# -O3: Maximum optimization for speed
# -mcpu=native: Generates instructions optimized specifically for your Apple M1 CPU
compile_command = ["clang++", "-O3", "-mcpu=native", "main.cpp", "-o", "main"]
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)

run_command = ["./main"]
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)

print(run_result.stdout)
```

### Why these flags?
* **`clang++`**: The native C++ compiler on macOS.
* **`-O3`**: Applies aggressive compiler optimizations (loop unrolling, vectorization, inline functions) to achieve maximum execution speed.
* **`-mcpu=native`**: Tells Clang to tailor the machine code specifically to the Apple M1 architecture capabilities.

In [7]:
# setting up the compile command and run command

compile_command = [
    "clang++",
    "-O3",  # Maximum optimization for speed
    "-mcpu=native",  # Target Apple M1 specific CPU capabilities
    "-flto",  # Enable Link-Time Optimization
    "main.cpp",  # Input file
    "-o",
    "main",  # Output binary executable name
]

run_command = ["./main"]

#### Main Task

In [19]:
# system_prompt and user_prompt

system_prompt = """
Your task is to convert Python code into high performance C++ code.
Respond only with C++ code. Do not provide any explanation other than occasional comments.
The C++ response needs to produce an identical output in the fastest possible time.
"""

def user_prompt(python):
    return f"""
Port this Python code to C++ with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called main.cpp and then compiled and executed; the compilation command is:
{compile_command}
Respond only with C++ code.
Python code to port:

```python
{python}
```
"""

In [9]:
# messages 

def messages(python):
    return [
        {'role':'system', 'content':system_prompt},
        {'role':'user', 'content':user_prompt(python)}
    ]

In [10]:
# write to main.cpp

def write_cpp(code):
    with open('main.cpp', 'w') as f:
        f.write(code)

In [21]:
# to port the code from Python to Cpp

def port(model, python):
    client = clients[model]
    response = client.chat.completions.create(
        model = model,
        messages = messages(python)
    )
    reply = response.choices[0].message.content
    reply = reply.replace('```cpp', '').replace('```', '')
    return reply

In [12]:
# to run the python code form string

def run_python(code):
    global_builtins = {"__builtins__":__builtins__}

    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer

    try:
        exec(code, global_builtins)
        output = buffer.getvalue()
    except Exception as e:
        output = f'Error: {e}'
    finally:
        sys.stdout = old_stdout
    
    return output

In [22]:
# to compile and run cpp

def compile_and_run(code):
    write_cpp(code)
    try:
        subprocess.run(compile_command, check=True, text=True, capture_output=True)
        result = subprocess.run(run_command, check=True, text=True, capture_output=True)
        return result.stdout
    except subprocess.CalledProcessError as e:
        return f'Error occured: \n{e.stderr}'

In [14]:
# python code as string

python_hard = """# Be careful to support large numbers

def lcg(seed, a=1664525, c=1013904223, m=2**32):
    value = seed
    while True:
        value = (a * value + c) % m
        yield value
        
def max_subarray_sum(n, seed, min_val, max_val):
    lcg_gen = lcg(seed)
    random_numbers = [next(lcg_gen) % (max_val - min_val + 1) + min_val for _ in range(n)]
    max_sum = float('-inf')
    for i in range(n):
        current_sum = 0
        for j in range(i, n):
            current_sum += random_numbers[j]
            if current_sum > max_sum:
                max_sum = current_sum
    return max_sum

def total_max_subarray_sum(n, initial_seed, min_val, max_val):
    total_sum = 0
    lcg_gen = lcg(initial_seed)
    for _ in range(20):
        seed = next(lcg_gen)
        total_sum += max_subarray_sum(n, seed, min_val, max_val)
    return total_sum

# Parameters
n = 10000         # Number of random numbers
initial_seed = 42 # Initial seed for the LCG
min_val = -10     # Minimum value of random numbers
max_val = 10      # Maximum value of random numbers

# Timing the function
import time
start_time = time.time()
result = total_max_subarray_sum(n, initial_seed, min_val, max_val)
end_time = time.time()

print("Total Maximum Subarray Sum (20 runs):", result)
print("Execution Time: {:.6f} seconds".format(end_time - start_time))
"""

In [16]:
# running python code from string

print(run_python(python_hard))

Total Maximum Subarray Sum (20 runs): 10980
Execution Time: 40.117140 seconds



In [23]:
# styling the UI with Gradio and CSS

from styles import CSS

with gr.Blocks(css=CSS, theme=gr.themes.Monochrome(), title=f"Port from Python to C++") as ui:
    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            python = gr.Code(
                label = "Python (original)",
                value = python_hard,
                language = "python",
                lines = 26
            )
        with gr.Column(scale=6):
            cpp = gr.Code(
                label = "C++ (generated)",
                value = "",
                language = "cpp",
                lines = 26
            )

    with gr.Row(elem_classes=["controls"]):
        python_run = gr.Button("Run Python", elem_classes=["run-btn", "py"])
        model = gr.Dropdown(models, value=models[0], show_label=False)
        convert = gr.Button(f"Port to C++", elem_classes = ["convert-btn"])
        cpp_run = gr.Button(f"Run C++", elem_classes = ["run-btn", "cpp"])
        
    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
                python_out = gr.TextArea(label="Python result", lines=8, elem_classes=["py-out"])
        with gr.Column(scale=6):
                cpp_out = gr.TextArea(label="C++ result", lines=8, elem_classes=["cpp-out"])

    python_run.click(fn=run_python, inputs=[python], outputs=[python_out])
    convert.click(fn=port, inputs=[model, python], outputs=[cpp])
    cpp_run.click(fn=compile_and_run, inputs=[cpp], outputs=[cpp_out])

ui.launch(inbrowser=True)


/var/folders/dv/bj8y54ys6czc90jxfd5b64gc0000gn/T/ipykernel_96875/3589094474.py:5: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=CSS, theme=gr.themes.Monochrome(), title=f"Port from Python to C++") as ui:


* Running on local URL:  http://127.0.0.1:7892
* To create a public link, set `share=True` in `launch()`.
